In [1]:
from local_pool_price import *
from local_pool_price import _build_curve_pool_local_price, _build_balancer_pool_local_price
import plotly.express as px

curve_pools = _build_curve_pool_local_price()

In [2]:
# uint256 fee = IBasePool(pool).getSwapFeePercentage();

# // Add swap fee to the price calculation.
# // Balancer Fee is in 1e18.
# // Scaling price back up to get to value representitive of one full asset swapped.
# price = (uint256(-assetDeltas[1]) * (1e18 * padUnit)) / (1e18 - fee);

In [3]:
balancer_pools = _build_balancer_pool_local_price()
balancer_pools

[TokenLocalPoolPriceDetails(calls=[<Call querySwapExactIn((address,(address,address,bool)[],uint256,uint256)[],address,bytes)(uint256[],address[],uint256[]) on 0x136f1E>, <Call querySwapExactIn((address,(address,address,bool)[],uint256,uint256)[],address,bytes)(uint256[],address[],uint256[]) on 0x136f1E>, <Call querySwapExactIn((address,(address,address,bool)[],uint256,uint256)[],address,bytes)(uint256[],address[],uint256[]) on 0x136f1E>], pool_name='GHO_USDC_USDT_boosted', pool_address='0x85B2b559bC2D21104C4DEFdd6EFcA8A20343361D'),
 TokenLocalPoolPriceDetails(calls=[<Call querySwap((bytes32,uint8,address,address,uint256,bytes),(address,bool,address,bool))(uint256) on 0xE39B5e>, <Call querySwap((bytes32,uint8,address,address,uint256,bytes),(address,bool,address,bool))(uint256) on 0xE39B5e>], pool_name='sUSDe_USDC_balancer_pool', pool_address='0xb819feeF8F0fcDC268AfE14162983A69f6BF179E')]

In [30]:
# import numpy as np


# def _build_curve_swap_fee_calls() -> list[Call]:
#     curve_pools = _build_curve_pool_local_price()
#     token_local_price_call_list = [t.calls for t in curve_pools]
#     token_local_price_calls = [c for calls in token_local_price_call_list for c in calls]
#     FEE_PRECISION = 1e10

#     def _compute_fee_as_portion(success: bool, fee: int):
#         if success:
#             return fee / FEE_PRECISION
#         return np.nan

#     pool_name_tuples = []
#     for c in token_local_price_calls:
#         pool_address = c.target
#         pool_name = c.returns[0][0].split("__")[0]

#         pool_name_tuples.append((pool_name, pool_address))
#     pool_name_tuples = set(pool_name_tuples)
#     pool_name_tuples

#     curve_fee_calls = [
#         Call(
#             pool,
#             ["fee()(uint256)"],
#             [(f"{pool_name}_fee", _compute_fee_as_portion)],
#         )
#         for pool_name, pool in pool_name_tuples
#     ]
#     return curve_fee_calls


# def _build_balancer_swap_fee_calls() -> list[Call]:

#     FEE_PRECISION = 1e18

#     def _compute_fee_as_portion(success: bool, fee: int):
#         if success:
#             return fee / FEE_PRECISION
#         return np.nan

#     # def _compute_fee_as_portion_agg(success: bool,args):
#     #     aggregateSwapFeePercentage, aggregateYieldFeePercentage = args
#     #     # return args
#     #     if success:
#     #         return aggregateSwapFeePercentage / FEE_PRECISION
#     #     return np.nan


#     balancer_fee_calls = [
#         Call(
#             "0x85B2b559bC2D21104C4DEFdd6EFcA8A20343361D",
#             ["getStaticSwapFeePercentage()(uint256)"],
#             [("GHO_USDC_USDT_boosted_fee", _compute_fee_as_portion)],
#         ),
#         Call(
#             "0xb819feeF8F0fcDC268AfE14162983A69f6BF179E",
#             ["getSwapFeePercentage()(uint256)"],
#             [("sUSDe_USDC_balancer_pool_fee", _compute_fee_as_portion )],
#         ),
#     ]

#     return balancer_fee_calls


# def _build_fee_calls():
#     curve_fee_calls = _build_curve_swap_fee_calls()
#     balancer_fee_calls = _build_balancer_swap_fee_calls()
#     return [*curve_fee_calls, *balancer_fee_calls]


# from mainnet_launch.data_fetching.get_state_by_block import (
#     identity_with_bool_success,
#     build_blocks_to_use,
#     get_state_by_one_block,
# )
# # get_state_by_one_block(_build_fee_calls(), ETH_CHAIN.client.eth.block_number, ETH_CHAIN)
# blocks = build_blocks_to_use(ETH_CHAIN)
# fee_df = get_raw_state_by_blocks(_build_fee_calls(), blocks, ETH_CHAIN)
# px.line(fee_df)

In [26]:
500000000000000000 / 1e18

0.5

In [28]:
100 * 50000000000000 / 1e18  # .5bps?

0.005

In [ ]:
USDC

'0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48'

In [ ]:
# the boosted pool has a differnet function singanture

In [ ]:
from mainnet_launch.data_fetching.get_state_by_block import identity_with_bool_success, build_blocks_to_use

token_local_price_call_list = [t.calls for t in curve_pools]
token_local_price_calls = [c for calls in token_local_price_call_list for c in calls]
FEE_PRECISION = 1e10
pool_name_tuples = []
for c in token_local_price_calls:
    pool_address = c.target
    pool_name = c.returns[0][0].split("__")[0]

    pool_name_tuples.append((pool_name, pool_address))
pool_name_tuples = set(pool_name_tuples)
pool_name_tuples

fee_calls = [
    Call(
        pool,
        ["fee()(uint256)"],
        [(f"{pool_name}_fee", identity_with_bool_success)],
    )
    for pool_name, pool in pool_name_tuples
]

# price = (dy * FEE_PRECISION) / (FEE_PRECISION - fee);
blocks = build_blocks_to_use(ETH_CHAIN)
fee_df = get_raw_state_by_blocks(_build_fee_calls(), blocks, ETH_CHAIN)

In [ ]:
px.line(100 * fee_df / FEE_PRECISION)

In [ ]:
from mainnet_launch.constants import *
from mainnet_launch.data_fetching.get_state_by_block import *
import numpy as np
from local_pool_price import *
import plotly.express as px


def build_get_token_price_in_quote(token_address: str, name: str) -> Call:
    return Call(
        ROOT_PRICE_ORACLE(ETH_CHAIN),
        ["getPriceInQuote(address,address)(uint256)", token_address, USDC],
        [(name, safe_normalize_6_with_bool_success)],
    )


# 0xb819feeF8F0fcDC268AfE14162983A69f6BF179E balacner pool without getSwqap

get_state_by_one_block(build_get_token_price_in_quote("", "spot USDC"), ETH_CHAIN.client.eth.block_number, eth_client)

# safe_price_calls = build_safe_price_calls()

# get_token_price_in_USDC_calls = [
#     build_get_token_price_in_quote(c[0], c[1] + "_to_USDC_via_rootPriceOracle") for c in stable_coin_tuples
# ]